# Classroom Attempts — Fresh Colab Validation

Student implementer and maintainer: YQ-HA, including post-course completions. Course methods: Tyler Shoemaker. Codex assisted with error explanations, repairs, restoration, and checks.

Run one cell at a time. These repaired classroom notebooks have NOT been executed end to end. Start with CPU A–H, then GPU section I. Historical results from the earlier project do not validate this version.


## A. Check Python and unpack the complete project

Use the full project ZIP supplied with the local delivery, including its seven data files.
A source-only GitHub ZIP is not a substitute for this archive.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import os
import sys
import time

print(sys.version)
assert (3, 12) <= sys.version_info[:2] < (3, 15), "This project requires Python 3.12–3.14."
with ZipFile("/content/interpretability-classroom-notebooks.zip") as archive:
    archive.extractall("/content")
os.chdir("/content/interpretability-classroom-notebooks")
# Keep the included reference run separate from this new run's results.
if any(Path("results").glob("session-*")):
    reference = Path("/content") / f"reference-results-{time.time_ns()}"
    Path("results").rename(reference)
    print("Historical reference results moved to:", reference)
print("Project:", Path.cwd())
print("Course data files:", len(list(Path("data").glob("*.npz"))) + len(list(Path("data").glob("*.parquet"))))


## B. Install dependencies in the cloud runtime

This does not install anything on your computer. The project keeps Transformers
in the validated 4.57 series; the exact installed versions are saved below.
If Colab requests a restart after installation, restart the session, then continue at C.
Do not repeat A in an existing runtime after generating results.


In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-r", "requirements.txt",
], check=True)
subprocess.run([
    sys.executable, "-m", "ipykernel", "install", "--user",
    "--name", "python3", "--display-name", "Classroom Attempts",
], check=True)


## C. Create the log helper and record this runtime

Re-run this cell after restarting the Python session. Each command saves its exit code,
elapsed time, and full log. A failed command stops the current cell after saving its log.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import subprocess
import sys
import time

os.chdir("/content/interpretability-classroom-notebooks")
ROOT = Path.cwd()
LOGS = ROOT / "results" / "validation"
LOGS.mkdir(parents=True, exist_ok=True)

def run_logged(name, args):
    started = datetime.now(timezone.utc).isoformat()
    before = time.monotonic()
    print("Starting:", name, flush=True)
    with (LOGS / f"{name}.log").open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            args, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
            env={**os.environ, "PYTHONUNBUFFERED": "1", "MPLBACKEND": "Agg"},
        )
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        returncode = process.wait()
    record = {
        "command": args, "started_utc": started,
        "elapsed_seconds": round(time.monotonic() - before, 2),
        "returncode": returncode,
        "status": "passed" if returncode == 0 else "failed",
    }
    (LOGS / f"{name}.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
    print(json.dumps(record, indent=2))
    assert returncode == 0, f"{name} failed. Download results and send back its log."

def run_lab(session):
    run_logged(f"session-{session:02d}", [
        sys.executable, "-u", "tools/run_notebook.py",
        "--session", str(session),
    ])

(LOGS / "pip-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    encoding="utf-8",
)
sources = sorted((ROOT / "notebooks").glob("*.ipynb")) + sorted((ROOT / "tools").glob("*.py")) + sorted((ROOT / "tests").glob("*.py"))
imported_versions = json.loads(subprocess.check_output([
    sys.executable, "-c",
    "import json, numpy, pandas, torch, transformers, sklearn; "
    "print(json.dumps({'numpy': numpy.__version__, 'pandas': pandas.__version__, "
    "'torch': torch.__version__, 'transformers': transformers.__version__, "
    "'scikit-learn': sklearn.__version__, 'cuda_available': torch.cuda.is_available()}))",
], text=True))
metadata = {
    "python": platform.python_version(), "platform": platform.platform(),
    "imported_versions": imported_versions,
    "sources_sha256": {str(path.relative_to(ROOT)): hashlib.sha256(path.read_bytes()).hexdigest() for path in sources},
}
(LOGS / "runtime.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Logs:", LOGS)


## D. Check integrity and six numerical tests

Expect the integrity report to say `status: passed`, followed by `Ran 6 tests` and `OK`.
This verifies file integrity and selected methods, not complete experiment execution.


In [ ]:
run_logged("integrity", [sys.executable, "tools/check_integrity.py", "--require-data"])
run_logged("unit-tests", [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"])


## E. First CPU milestone — sessions 05 and 06

These use cached course data and do not download a language model.
Wait for each session to print `Saved executed notebook`.
If either fails, jump to H, download the evidence, and send it back before continuing.


In [ ]:
for session in [5, 6]:
    run_lab(session)


## F. Extend CPU validation — session 07

This performs 660 cross-validation fits, followed by the remaining analyses.
CPU work may take substantially longer than sessions 05 and 06.


In [ ]:
run_lab(7)


## G. Extend CPU validation — sessions 01–04

The first run downloads GPT-2. The following runs reuse its model cache.
A model-download error is a distinct issue from a numerical or notebook error.


In [ ]:
for session in [1, 2, 3, 4]:
    run_lab(session)


## H. Download results — use after a milestone, failure, or before changing runtime

Run this even when an earlier session fails. Completed notebooks, figures, tables,
runtime metadata, and error logs are included. A failed session may have partial
figure/table files; its JSON status and log determine whether it completed.
Also save this Colab notebook using File → Download → Download .ipynb.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
from google.colab import files

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output = shutil.make_archive(
    f"/content/classroom-validation-{stamp}", "zip",
    root_dir="/content/interpretability-classroom-notebooks", base_dir="results",
)
print(output, "bytes:", Path(output).stat().st_size)
files.download(output)


## I. GPU phase — session 08 only

1. Download the CPU results using H.
2. Choose Runtime → Change runtime type → T4 GPU (or another free GPU).
   If no free GPU is available, retain the CPU results and report session 08 as pending.
3. On the new runtime, upload the project ZIP and repeat A, B, and C.
4. Run the next cell, then H to download GPU results.

Attempt 7 already uses the float16 setting intended for T4. No source patch is
required. Keep the full 2,048-token generation budget. An earlier project run reached
that budget and analyzed a truncated trace; a successful analysis does not imply
that the model completed its reasoning.


In [ ]:
import torch

assert torch.cuda.is_available(), "Select a GPU runtime before running session 08."
gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)

notebook_path = ROOT / "notebooks" / "Attempt 7.ipynb"
device_record = {
    "gpu": gpu_name, "torch": torch.__version__,
    "model_dtype": "float16", "source_patch_applied": False,
    "notebook_sha256": hashlib.sha256(notebook_path.read_bytes()).hexdigest(),
}
(LOGS / "session-08-device.json").write_text(json.dumps(device_record, indent=2), encoding="utf-8")
run_lab(8)
